# 05_Neo4j_Load — Chargement du graphe

Reprend la logique déjà validée dans `entraide-vm2/scripts/load_neo4j.py`
(2 bugs corrigés lors de cette validation croisée, appliqués ici dès le
départ) :
- **Direction de relation cohérente** : `Commune-[:DANS]->Delegation-[:DANS]->Region`
  partout (un bug de direction opposée entre la création de la hiérarchie et
  le rattachement des centres avait fait échouer silencieusement TOUT le
  chargement des nœuds Centre lors de la première validation).
- **Pattern MERGE unique** pour la hiérarchie géo (scope correct par parent,
  pas de collision entre deux délégations homonymes de régions différentes).

Idempotent (`DETACH DELETE` puis rechargement complet).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from etl_lib.ontology import INSTITUTION_MAP, POPULATION_CIBLES
from etl_lib.io_utils import load_jsonl

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
RELATIONS_DIR = Path.cwd().parent / "data" / "relations"

services = load_jsonl(PROCESSED_DIR / "services_unified.jsonl")
centres = load_jsonl(PROCESSED_DIR / "centres_normalized.jsonl")
programmes = load_jsonl(PROCESSED_DIR / "programmes_2027.jsonl")
faqs = load_jsonl(PROCESSED_DIR / "faq_aos.jsonl")
rel_c2s = load_jsonl(RELATIONS_DIR / "relations_centre_service.jsonl")
rel_p2i = load_jsonl(RELATIONS_DIR / "relations_programme_institution.jsonl")
rel_p2pop = load_jsonl(RELATIONS_DIR / "relations_programme_population.jsonl")
rel_s2pop = load_jsonl(RELATIONS_DIR / "relations_service_population.jsonl")
rel_f2p = load_jsonl(RELATIONS_DIR / "relations_faq_programme.jsonl")

print(f"services={len(services)} centres={len(centres)} programmes={len(programmes)} faq={len(faqs)}")
print(f"rel centre-service={len(rel_c2s)} rel prog-inst={len(rel_p2i)} rel prog-pop={len(rel_p2pop)} "
      f"rel svc-pop={len(rel_s2pop)} rel faq-prog={len(rel_f2p)}")


services=61 centres=3328 programmes=7 faq=8
rel centre-service=24518 rel prog-inst=13 rel prog-pop=8 rel svc-pop=61 rel faq-prog=0


In [2]:
# === FAQ_EXTRA_INGESTION_BLOCK ===
faq_extra_service = load_jsonl(PROCESSED_DIR / "faq_extra_service.jsonl")
faq_extra_general = load_jsonl(PROCESSED_DIR / "faq_extra_general.jsonl")
institution_enrich = load_jsonl(PROCESSED_DIR / "institution_enrich.jsonl")
print(f"data_faq: {len(faq_extra_service)} FAQ service, {len(faq_extra_general)} FAQ generiques, "
      f"{len(institution_enrich)} profils Institution")


data_faq: 58 FAQ service, 55 FAQ generiques, 8 profils Institution


In [3]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "changeme-neo4j")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Connecte a {NEO4J_URI}")


Connecte a bolt://localhost:27687


## 1. Reset + contraintes

In [4]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

constraints = [
    ("centre_id", "Centre", "id"), ("service_id", "Service", "id"),
    ("programme_id", "Programme", "id"), ("faq_id", "FAQ", "id"),
    ("institution_code", "Institution", "code"), ("population_code", "PopulationCible", "code"),
    ("region_name", "Region", "name"),
]
with driver.session() as session:
    for name, label, prop in constraints:
        session.run(f"CREATE CONSTRAINT {name} IF NOT EXISTS FOR (n:{label}) REQUIRE n.{prop} IS UNIQUE")
print(f"Base videe, {len(constraints)} contraintes creees")


Base videe, 7 contraintes creees


## 2. Hiérarchie géographique — Region -> Delegation -> Commune

In [5]:
centre_geo = []
with driver.session() as session:
    for c in centres:
        if not (c["region"] and c["delegation"] and c["commune"]):
            continue
        session.run(
            "MERGE (r:Region {name: $region}) "
            "MERGE (d:Delegation {name: $delegation})-[:DANS]->(r) "
            "MERGE (c:Commune {name: $commune})-[:DANS]->(d)",
            region=c["region"], delegation=c["delegation"], commune=c["commune"],
        )
        centre_geo.append(c)
print(f"Hierarchie chargee pour {len(centre_geo)} centres")


Hierarchie chargee pour 3328 centres


## 3. Nœuds Centre (rattachés à leur commune — même sens de relation que ci-dessus)

In [6]:
with driver.session() as session:
    for i, c in enumerate(centre_geo):
        session.run(
            "MATCH (com:Commune {name: $commune})-[:DANS]->(:Delegation {name: $delegation})-[:DANS]->(:Region {name: $region}) "
            "MERGE (ctr:Centre {id: $id}) SET ctr += $props "
            "MERGE (ctr)-[:ABRITE_DANS]->(com)",
            commune=c["commune"], delegation=c["delegation"], region=c["region"], id=c["id"],
            props={
                "nom": c["nom"], "adresse": c["adresse"], "activite": c["activite"],
                "milieu": c["milieu"], "propriete": c["propriete"],
                "capacite": c["capacite"], "superficie": c["superficie"],
                "region": c["region"], "delegation": c["delegation"], "commune": c["commune"],
                "institutions": c["institutions"], "personnes_cibles": c.get("personnes_cibles", ""),
            },
        )
        if (i + 1) % 1000 == 0:
            print(f"  {i + 1}/{len(centre_geo)}...")

with driver.session() as session:
    n = session.run("MATCH (c:Centre) RETURN count(c) AS n").single()["n"]
print(f"Noeuds Centre reellement crees (verifie en base) : {n}")
assert n == len(centre_geo), "Le nombre de Centre en base ne correspond pas -- probable bug de direction de relation"


  1000/3328...


  2000/3328...


  3000/3328...


Noeuds Centre reellement crees (verifie en base) : 3328


## 4. Nœuds Service / Institution / PopulationCible / Programme / FAQ

In [7]:
with driver.session() as session:
    for s in services:
        session.run(
            "MERGE (s:Service {id: $id}) SET s += $props", id=s["id"],
            props={
                "service_ar": s["service_ar"], "service_fr": s.get("service_fr", ""),
                "categorie": s["categorie"], "description_ar": s["description_ar"],
                "conditions_ar": s["conditions_ar"], "documents_ar": s["documents_ar"],
                "institutions": s["institutions"], "eps_fallback": s["eps_fallback"],
            },
        )
    for code, data in INSTITUTION_MAP.items():
        session.run("MERGE (i:Institution {code: $code}) SET i += $props", code=code,
                     props={"ar": data["ar"], "fr": data["fr"], "polyvalence": data["polyvalence"]})
    for code, data in POPULATION_CIBLES.items():
        session.run("MERGE (p:PopulationCible {code: $code}) SET p += $props", code=code,
                     props={"ar": data["ar"], "fr": data["fr"], "priorite": data["priorite"]})
    for p in programmes:
        session.run("MERGE (p:Programme {id: $id}) SET p += $props", id=p["id"],
                     props={"code": p["code"], "titre_ar": p["titre_ar"], "titre_fr": p["titre_fr"],
                            "budget": p.get("budget", 0), "population_cible": p.get("population_cible", ""),
                            "institutions_liees": p.get("institutions_liees", [])})
    for f in faqs:
        session.run("MERGE (f:FAQ {id: $id}) SET f += $props", id=f["id"],
                     props={"question_ar": f["question_ar"], "reponse_ar": f["reponse_ar"],
                            "section_ar": f.get("section_ar", ""), "programme": f.get("programme", "")})

print(f"{len(services)} Service, {len(INSTITUTION_MAP)} Institution, {len(POPULATION_CIBLES)} PopulationCible, "
      f"{len(programmes)} Programme, {len(faqs)} FAQ crees")


61 Service, 14 Institution, 6 PopulationCible, 7 Programme, 8 FAQ crees


In [8]:
# === FAQ_EXTRA_INGESTION_BLOCK ===
# FAQ issues de data_faq (familles A+C) -- memes proprietes que les FAQ AOS
# d'origine + category/family pour tracabilite. Lien par SIGNAUX CONFIANTS
# uniquement (population/institution extraits par regex/keyword, cf
# etl_lib.ontology) -- pas de lien FAQ->Service invente : un matching
# semantique fiable question<->service releve du retrieval (Qdrant), pas
# d'une arete de graphe figee sur un score incertain.
with driver.session() as session:
    for f in faq_extra_service + faq_extra_general:
        session.run(
            "MERGE (faq:FAQ {id: $id}) SET faq += $props",
            id=f["id"],
            props={
                "question_ar": f["question_ar"], "reponse_ar": f["reponse_ar"],
                "category": f.get("category", ""), "family": f.get("family", ""),
                "source": "data_faq",
            },
        )
        for pcode in f.get("population_cible", []):
            session.run(
                "MATCH (faq:FAQ {id: $id}), (p:PopulationCible {code: $pcode}) MERGE (faq)-[:CIBLE]->(p)",
                id=f["id"], pcode=pcode,
            )
        for icode in f.get("institutions", []):
            session.run(
                "MATCH (faq:FAQ {id: $id}), (i:Institution {code: $icode}) MERGE (faq)-[:CONCERNE]->(i)",
                id=f["id"], icode=icode,
            )

    for ip in institution_enrich:
        session.run(
            "MATCH (i:Institution {code: $code}) SET i.profile_ar = $profile, i.profile_source = $source_id",
            code=ip["institution_code"], profile=ip["reponse_ar"], source_id=ip["source_id"],
        )

n_faq_extra = len(faq_extra_service) + len(faq_extra_general)
print(f"{n_faq_extra} FAQ (data_faq) creees/mises a jour, "
      f"{len(institution_enrich)} profils Institution enrichis")


113 FAQ (data_faq) creees/mises a jour, 8 profils Institution enrichis


## 5. Relations (OFFRE, CIBLE, S_APPUIE_SUR, REPOND_A)

In [9]:
with driver.session() as session:
    for i, r in enumerate(rel_c2s):
        session.run(
            "MATCH (c:Centre {id: $cid}), (s:Service {id: $sid}) "
            "MERGE (c)-[rel:OFFRE]->(s) SET rel.score = $score, rel.method = $method",
            cid=r["source_id"], sid=r["target_id"], score=r["score"], method=r["method"],
        )
        if (i + 1) % 5000 == 0:
            print(f"  OFFRE {i + 1}/{len(rel_c2s)}...")

    for r in rel_s2pop:
        session.run("MATCH (s:Service {id: $sid}), (p:PopulationCible {code: $pcode}) MERGE (s)-[:CIBLE]->(p)",
                     sid=r["source_id"], pcode=r["target_id"])
    for r in rel_p2pop:
        session.run("MATCH (p:Programme {id: $pid}), (pop:PopulationCible {code: $pcode}) MERGE (p)-[:CIBLE]->(pop)",
                     pid=r["source_id"], pcode=r["target_id"])
    for r in rel_p2i:
        session.run("MATCH (p:Programme {id: $pid}), (i:Institution {code: $icode}) MERGE (p)-[:S_APPUIE_SUR]->(i)",
                     pid=r["source_id"], icode=r["target_id"])
    for r in rel_f2p:
        session.run("MATCH (f:FAQ {id: $fid}), (p:Programme {id: $pid}) MERGE (f)-[:REPOND_A]->(p)",
                     fid=r["source_id"], pid=r["target_id"])

print(f"{len(rel_c2s)} OFFRE, {len(rel_s2pop) + len(rel_p2pop)} CIBLE, {len(rel_p2i)} S_APPUIE_SUR, {len(rel_f2p)} REPOND_A crees")


  OFFRE 5000/24518...


  OFFRE 10000/24518...


  OFFRE 15000/24518...


  OFFRE 20000/24518...


24518 OFFRE, 69 CIBLE, 13 S_APPUIE_SUR, 0 REPOND_A crees


## 6. Vérification finale

In [10]:
with driver.session() as session:
    print("Noeuds:")
    for r in session.run("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS c ORDER BY c DESC"):
        print(f"  {r['label']:20}: {r['c']}")
    print("Relations:")
    for r in session.run("MATCH ()-[r]->() RETURN type(r) AS t, count(*) AS c ORDER BY c DESC"):
        print(f"  {r['t']:20}: {r['c']}")

    avg = session.run(
        "MATCH (c:Centre)-[r:OFFRE]->(:Service) WITH c, count(r) AS n RETURN avg(n) AS moyenne, max(n) AS maximum"
    ).single()
    print(f"\nMoyenne services/centre (verifie en base): {avg['moyenne']:.2f} (cible <= 8)")

driver.close()
print("\n\u2705 05_Neo4j_Load termine.")


Noeuds:
  Centre              : 3328
  Commune             : 1086
  FAQ                 : 121
  Delegation          : 82
  Service             : 61
  Institution         : 14
  Region              : 12
  Programme           : 7
  PopulationCible     : 6
Relations:
  OFFRE               : 24518
  ABRITE_DANS         : 3328
  DANS                : 1168
  CIBLE               : 242
  CONCERNE            : 155
  S_APPUIE_SUR        : 12

Moyenne services/centre (verifie en base): 7.37 (cible <= 8)

✅ 05_Neo4j_Load termine.
